<a href="https://colab.research.google.com/github/OdysseusPolymetis/initiation_ia/blob/main/CLIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# StyleCLIP : modifier une image avec du texte

Dans ce notebook, on ne génère pas seulement une image : on apprend à la **modifier** à partir d'une consigne textuelle.

## Idée principale
On combine deux outils :

- **StyleGAN** : produit une image à partir d'un vecteur latent ;
- **CLIP** : mesure si une image correspond bien à un texte.

## Objectif
On part d'une image de visage, puis on essaie de la rapprocher d'une description textuelle, par exemple :

- "a smiling person"
- "a person with red hair"
- "a person with glasses"

## Ce qu'on veut comprendre
- ce qu'apporte CLIP ;
- en quoi StyleCLIP diffère de StyleGAN2 et StyleGAN3 ;
- comment un texte peut guider une modification d'image.

## Où se situe StyleCLIP dans le cours ?

Jusqu'ici :

- **StyleGAN2** : générer une image ;
- **StyleGAN3** : faire évoluer une image de façon plus cohérente.

Maintenant :

- **StyleCLIP** : modifier une image selon une consigne textuelle.

## Idée simple
Avec StyleGAN2 et StyleGAN3, le contrôle se faisait surtout avec :
- le modèle ;
- le seed ;
- la troncature.

Avec StyleCLIP, on ajoute un nouveau type de contrôle :
- le **texte**.

## Ce que fait réellement StyleCLIP

StyleCLIP n'est pas un modèle text-to-image natif comme Stable Diffusion.

Il fonctionne autrement :

1. StyleGAN sait produire des visages ;
2. CLIP sait rapprocher des images et des textes ;
3. on modifie le vecteur latent pour que l'image produite soit plus compatible avec le texte.

## Formule simple

**latent de départ -> image -> comparaison avec le texte -> modification du latent**

In [ ]:
!nvidia-smi || true

import sys
import platform
import torch

print("Python :", sys.version)
print("Plateforme :", platform.platform())
print("Torch version :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

## Installation minimale

On installe ici seulement les bibliothèques utiles pour :

- charger StyleGAN ;
- utiliser CLIP ;
- afficher les images.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y build-essential ninja-build
!pip -q install numpy pillow matplotlib ftfy regex tqdm
!pip -q install git+https://github.com/openai/CLIP.git

## Récupérer le code StyleGAN2-ADA

Pourquoi StyleGAN2 ici ?

Parce que StyleCLIP a été conçu pour manipuler des images produites par StyleGAN.
Pour un notebook débutant, on va garder une base simple :
- un générateur StyleGAN préentraîné ;
- un vecteur latent ;
- un texte ;
- une optimisation guidée par CLIP.

In [ ]:
!rm -rf /content/stylegan2-ada-pytorch
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git /content/stylegan2-ada-pytorch
%cd /content/stylegan2-ada-pytorch

## Charger les bibliothèques utiles

Nous allons charger :

- le générateur StyleGAN ;
- CLIP ;
- des outils d'affichage ;
- des outils d'optimisation.

In [ ]:
import os
import numpy as np
import torch
from torch import optim
from PIL import Image
import matplotlib.pyplot as plt

import clip
import dnnlib
import legacy

## Choisir un modèle StyleGAN préentraîné

On utilise ici un modèle FFHQ :
- il génère des visages photoréalistes ;
- il servira de base pour l'édition textuelle.

## Idée importante
Le texte ne remplace pas le modèle.

Le texte guide seulement la transformation **dans ce que le modèle sait déjà produire**.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NETWORK_URL = "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl"

with dnnlib.util.open_url(NETWORK_URL) as f:
    G = legacy.load_network_pkl(f)["G_ema"].to(device)

print("Générateur chargé.")
print("z_dim =", G.z_dim)
print("c_dim =", G.c_dim)

## Charger CLIP

CLIP est un modèle qui place :
- les images ;
- les textes

dans un espace commun.

Cela permet de mesurer si une image "correspond" à un texte.

In [ ]:
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("CLIP chargé.")

## Générer une image de départ

On commence par un visage généré à partir d'un seed.

## Pourquoi ?
Parce qu'il faut un point de départ avant de pouvoir éditer l'image.

In [ ]:
seed = 42
truncation = 0.7

rng = np.random.RandomState(seed)
z0 = torch.from_numpy(rng.randn(1, G.z_dim)).to(device=device, dtype=torch.float32)

label = torch.zeros([1, G.c_dim], device=device)

with torch.no_grad():
    img = G(z0, label, truncation_psi=truncation, noise_mode="const")
    img_uint8 = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    img_pil = Image.fromarray(img_uint8[0].cpu().numpy(), "RGB")

plt.figure(figsize=(5,5))
plt.imshow(img_pil)
plt.title("Image de départ")
plt.axis("off")
plt.show()

## Choisir une consigne textuelle

On va maintenant donner un texte au système.

## Conseils pour débuter
Mieux vaut choisir des consignes simples, comme :

- "a smiling person"
- "a person with red hair"
- "a person with glasses"

Certaines consignes fonctionnent mieux que d'autres.

In [ ]:
target_text = "an angry person"
print("Texte cible :", target_text)

## Préparer le texte pour CLIP

CLIP ne lit pas directement la phrase telle quelle :
il la transforme en représentation numérique, comme il le fait aussi pour les images.

In [ ]:
text_tokens = clip.tokenize([target_text]).to(device)

with torch.no_grad():
    text_features = clip_model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

## Idée de l'optimisation

Maintenant, on ne change pas directement les pixels de l'image.

On change le **vecteur latent**.

À chaque étape :
1. on génère une image à partir du latent courant ;
2. CLIP compare cette image avec le texte ;
3. on modifie le latent pour améliorer la correspondance.

In [ ]:
z = z0.clone().detach().requires_grad_(True)
optimizer = optim.Adam([z], lr=0.05)

num_steps = 60
snapshots = []

## Fonction utilitaire : convertir une image StyleGAN en image CLIP

StyleGAN produit des tenseurs adaptés à la génération.
CLIP attend un autre format.

On ajoute donc une petite fonction de conversion.

In [ ]:
import torch.nn.functional as F

def stylegan_tensor_to_clip_input(img_tensor):
    # img_tensor : [1, 3, H, W] dans la plage approximative [-1, 1]
    x = (img_tensor + 1) / 2.0
    x = x.clamp(0, 1)
    x = F.interpolate(x, size=(224, 224), mode="bicubic", align_corners=False)

    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=x.device).view(1,3,1,1)
    std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=x.device).view(1,3,1,1)

    x = (x - mean) / std
    return x

## Lancer l'optimisation guidée par CLIP

On va faire plusieurs étapes.

### Ce qu'il faut observer
- l'image change progressivement ;
- le visage essaie de se rapprocher du texte ;
- certaines modifications sont plus faciles que d'autres.

In [ ]:
for step in range(num_steps):
    optimizer.zero_grad()

    img = G(z, label, truncation_psi=truncation, noise_mode="const")
    clip_input = stylegan_tensor_to_clip_input(img)

    image_features = clip_model.encode_image(clip_input)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)

    similarity = (image_features * text_features).sum()
    loss = -similarity

    loss.backward()
    optimizer.step()

    if step % 15 == 0 or step == num_steps - 1:
        with torch.no_grad():
            preview = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
            preview_pil = Image.fromarray(preview[0].cpu().numpy(), "RGB")
            snapshots.append((step, preview_pil.copy(), float(similarity.item())))
            print(f"step={step:03d} | similarity={similarity.item():.4f}")

## Observer l'évolution de l'image

On affiche maintenant plusieurs étapes de l'optimisation.

In [ ]:
plt.figure(figsize=(16,4))

for i, (step, img_snap, sim) in enumerate(snapshots, start=1):
    plt.subplot(1, len(snapshots), i)
    plt.imshow(img_snap)
    plt.title(f"step {step}\nsim={sim:.3f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
with torch.no_grad():
    final_img = G(z, label, truncation_psi=truncation, noise_mode="const")
    final_uint8 = (final_img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    final_pil = Image.fromarray(final_uint8[0].cpu().numpy(), "RGB")

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(img_pil)
plt.title("Avant")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(final_pil)
plt.title("Après")
plt.axis("off")

plt.tight_layout()
plt.show()

## Charger sa propre image

Vous pouvez maintenant téléverser votre propre photo avec `files.upload()`.

### Important
Dans ce notebook, l'image téléversée peut être :
- affichée ;
- analysée avec CLIP ;
- comparée à un texte.

Mais pour la **modifier avec StyleCLIP**, il faudrait d'abord la **projeter dans l'espace latent de StyleGAN**.  
Cette étape s'appelle **l'inversion**.

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Fichiers téléversés :", list(uploaded.keys()))

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

# On garde seulement les fichiers image classiques
image_extensions = (".png", ".jpg", ".jpeg", ".webp")
uploaded_images = [f for f in uploaded.keys() if f.lower().endswith(image_extensions)]

if len(uploaded_images) == 0:
    raise ValueError("Aucune image valide n'a été téléversée.")

user_image_path = uploaded_images[0]
print("Image sélectionnée :", user_image_path)

user_img = Image.open(user_image_path).convert("RGB")
print("Taille originale :", user_img.size)

plt.figure(figsize=(6,6))
plt.imshow(user_img)
plt.title("Image téléversée par l'étudiant")
plt.axis("off")
plt.show()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def center_crop_to_square(img):
    width, height = img.size
    side = min(width, height)

    left = (width - side) // 2
    top = (height - side) // 2
    right = left + side
    bottom = top + side

    return img.crop((left, top, right, bottom))

user_img_cropped = center_crop_to_square(user_img)
user_img_square = user_img_cropped.resize((512, 512))

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(user_img_cropped)
plt.title("Crop centré carré")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(user_img_square)
plt.title("Crop carré redimensionné en 512x512")
plt.axis("off")

plt.tight_layout()
plt.show()

## Utiliser CLIP sur l'image téléversée

Nous avons maintenant une image réelle, recadrée et redimensionnée.

### Ce que l'on peut faire tout de suite
On peut demander à CLIP de comparer cette image à plusieurs descriptions textuelles.

### Pourquoi c'est intéressant ?
Cela montre que CLIP peut fonctionner sur des images réelles, pas seulement sur des images générées par StyleGAN.

In [ ]:
import torch
import clip
from PIL import Image
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()

print("CLIP chargé sur :", device)

## Préparer l'image pour CLIP

CLIP attend une image prétraitée d'une certaine manière.

On convertit donc l'image croppée dans le format attendu par le modèle.

In [ ]:
clip_input_image = clip_preprocess(user_img_square).unsqueeze(0).to(device)

print("Shape du tenseur image :", clip_input_image.shape)

## Choisir plusieurs descriptions textuelles

On va proposer plusieurs phrases, puis demander à CLIP laquelle semble le mieux correspondre à l'image.

### Remarque
Les descriptions peuvent être simples ou plus précises.

In [ ]:
candidate_texts = [
    "a portrait of a person",
    "a smiling person",
    "a person with glasses",
    "a serious person",
    "a close-up face photo",
    "a selfie",
]

print("Descriptions testées :")
for t in candidate_texts:
    print("-", t)

## Convertir les textes en représentations numériques

Comme pour les images, CLIP transforme les textes en vecteurs.
Il peut ensuite comparer les vecteurs image et texte.

In [ ]:
text_tokens = clip.tokenize(candidate_texts).to(device)

## Calculer les similarités image / texte

CLIP donne ici un score pour chaque texte.

Plus le score est élevé, plus le modèle considère que le texte correspond bien à l'image.

In [ ]:
with torch.no_grad():
    logits_per_image, logits_per_text = clip_model(clip_input_image, text_tokens)
    probs = logits_per_image.softmax(dim=-1).cpu().numpy()[0]

In [ ]:
results = list(zip(candidate_texts, probs))
results = sorted(results, key=lambda x: x[1], reverse=True)

print("Résultats :")
for text, score in results:
    print(f"{text:30s} -> {score:.4f}")